##Training

In [1]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python
from datasets import load_dataset
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tqdm import tqdm

c:\Users\theke\Documents\Code\bharatanatyam-mudra-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')
from huggingface_hub import login
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from datasets import load_dataset

dataset = load_dataset("Samarth0710/bharatanatyam-mudra-dataset")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'gesture_type', 'label_id'],
        num_rows: 28431
    })
})


In [4]:
data = dataset["train"]

In [5]:
# Labels
labels = sorted(set(data["label"]))
label_to_id = {l: i for i, l in enumerate(labels)}

In [6]:
# MediaPipe
MODEL_PATH = "googletutorial/hand_landmarker.task"


In [7]:
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)
detector = vision.HandLandmarker.create_from_options(options)

In [8]:
def normalize_hand(hand_landmarks):
    # print('normalize hand')
    pts = np.array([[lm.x, lm.y] for lm in hand_landmarks])

    # Normalize by translation
    pts -= pts[0]

    # Normalize by scale
    scale = np.linalg.norm(pts[9])
    if scale > 1e-6:
        pts /= scale

    # Normalize by rotation: align wrist->middle MCP vector to point straight up (0, -1) ---
    ref = pts[9]
    angle = np.arctan2(ref[0], -ref[1])

    cos_a, sin_a = np.cos(angle), np.sin(angle)
    R = np.array([
        [cos_a, -sin_a],
        [sin_a,  cos_a]
    ])

    pts = pts @ R.T

    return pts.flatten()

def extract_features(image):
    img = np.array(image.convert("RGB"))
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    results = detector.detect(mp_image)

    if not results.hand_landmarks:
        return None
    feats = []

    for hand in results.hand_landmarks[:2]:
        feats.append(normalize_hand(hand))
        # for lm in hand:
        #     feats.extend([lm.x, lm.y, lm.z])

    # while len(feats) < 126:
    #     feats.extend([0]*63)
    while len(feats) < 2:
        feats.append(np.zeros(42))

    return np.concatenate(feats)
# Feature extraction
# def extract_features(image):
#     img = np.array(image.convert("RGB"))
#     mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
#     res = detector.detect(mp_image)

#     if not res.hand_landmarks:
#         return None

#     # TODO: Do the normalization here
#     feats = []

#     for hand in res.hand_landmarks[:2]:
#         for lm in hand:
#             # TODO: try just training on the x and y
#             feats.extend([lm.x, lm.y, lm.z])

#     while len(feats) < 126:
#         feats.extend([0]*63)

#     return np.array(feats)

In [9]:
# Build dataset
X, y = [], []

for sample in tqdm(data):
    feat = extract_features(sample["image"])
    if feat is None:
        continue
    # print(feat)
    X.append(feat)
    y.append(label_to_id[sample["label"]])

X = np.array(X)
y = np.array(y)

100%|██████████| 28431/28431 [07:31<00:00, 62.97it/s]


In [10]:
# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y
)


In [16]:
# Model
model = keras.Sequential([
    keras.layers.Input(shape=(84,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(len(labels), activation="softmax")
])

In [17]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
# Train
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=75)

Epoch 1/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 1s 917us/step - accuracy: 0.4303 - loss: 2.1498 - val_accuracy: 0.5994 - val_loss: 1.5012
Epoch 2/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step - accuracy: 0.6349 - loss: 1.3179 - val_accuracy: 0.6979 - val_loss: 1.1540
Epoch 3/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step - accuracy: 0.7014 - loss: 1.0687 - val_accuracy: 0.7402 - val_loss: 0.9773
Epoch 4/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 800us/step - accuracy: 0.7404 - loss: 0.9200 - val_accuracy: 0.7766 - val_loss: 0.8476
Epoch 5/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 814us/step - accuracy: 0.7630 - loss: 0.8164 - val_accuracy: 0.8024 - val_loss: 0.7467
Epoch 6/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - accuracy: 0.7860 - loss: 0.7204 - val_accuracy: 0.8159 - val_loss: 0.6884
Epoch 7/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 803us/step - accuracy: 0.8017 - loss: 0.6528 - val_accuracy: 0.8265 - val_loss: 0.6386
Epoch 8/75
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 805us/step - accuracy: 0.8180 - loss: 0.6074 - 

In [19]:
# Save
model.save("mudra_model.keras")

In [17]:
import json
with open("labels.json", "w") as f:
    json.dump(label_to_id, f)

In [24]:
'''
# realtime_mudra.py

import cv2
import mediapipe as mp
import numpy as np
import json
from tensorflow import keras
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python

# Load model
model = keras.models.load_model("mudra_model.keras")

with open("labels.json") as f:
    label_to_id = json.load(f)

id_to_label = {v:k for k,v in label_to_id.items()}

# MediaPipe
model_path = "/content/hand_landmarker (3).task"

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model_path),
    num_hands=2,
    running_mode=vision.RunningMode.VIDEO
)

detector = vision.HandLandmarker.create_from_options(options)

# Camera
cap = cv2.VideoCapture(0)
timestamp = 0

def extract_features(results):
    feats = []

    for hand in results.hand_landmarks[:2]:
        for lm in hand:
            feats.extend([lm.x, lm.y, lm.z])

    while len(feats) < 126:
        feats.extend([0]*63)

    return np.array(feats)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    results = detector.detect_for_video(mp_image, timestamp)
    timestamp += 1

    label_text = "No hand"

    if results.hand_landmarks:
        features = extract_features(results).reshape(1, -1)

        pred = model.predict(features, verbose=0)
        class_id = np.argmax(pred)
        confidence = np.max(pred)

        if confidence > 0.6:
            label_text = f"{id_to_label[class_id]} ({confidence:.2f})"

    cv2.putText(frame, label_text, (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Mudra Classifier", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [19]:
import numpy as np
import mediapipe as mp
import json
import cv2
from PIL import Image
from tensorflow import keras
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python
# from google.colab.patches import cv2_imshow

#load model
model = keras.models.load_model("mudra_model.keras")

with open("labels.json") as f:
    label_to_id = json.load(f)

id_to_label = {v: k for k, v in label_to_id.items()}

#mediapipe setup
# MODEL_PATH = "/content/hand_landmarker (3).task"

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

#feature extraction
def extract_features(results):
    feats = []

    for hand in results.hand_landmarks[:2]:
        for lm in hand:
            feats.extend([lm.x, lm.y, lm.z])

    while len(feats) < 126:
        feats.extend([0]*63)

    return np.array(feats, dtype=np.float32)

#load the image
image_path = "googletutorial/handimage.jpg"

pil_image = Image.open(image_path)
frame = np.array(pil_image.convert("RGB"))

h, w, _ = frame.shape

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=frame
)

results = detector.detect(mp_image)

#predict and draw bounding box
if results.hand_landmarks:

    features = extract_features(results).reshape(1, -1)

    preds = model.predict(features, verbose=0)
    class_id = np.argmax(preds)
    confidence = np.max(preds)

    label_text = f"{id_to_label[class_id]} ({confidence:.2f})"

    for hand_landmarks in results.hand_landmarks:

        xs = [lm.x for lm in hand_landmarks]
        ys = [lm.y for lm in hand_landmarks]

        x_min = int(min(xs) * w)
        y_min = int(min(ys) * h)
        x_max = int(max(xs) * w)
        y_max = int(max(ys) * h)

        # Draw bounding box
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0,255,0), 2)

        # Draw label
        cv2.putText(frame, label_text,
                    (x_min, y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0,150,0), 2)

else:
    print("No hand detected")

scale = 0.5
resized_frame = cv2.resize(frame, None, fx=scale, fy=scale)

# Display
# from google.colab.patches import cv2_imshow
# cv2_imshow()
cv2.imshow('Landmarks', cv2.cvtColor(resized_frame, cv2.COLOR_RGB2BGR))
key = cv2.waitKey(0)